# **Correction · TP6 | Fonctions de hachage**

*Corrigé rédigé en l'absence de corrigé officiel. Les réponses ont été vérifiées en Python ; les cellules qui lisent un fichier compagnon (`message1.bin`, `message2.bin`, `plane.jpg`, `ship.jpg`, les deux listes de mots de passe) n'ont pas pu être exécutées ici et sont à lancer dans le dossier du TP.*

## 6.1. Hachage cryptographique

Une fonction de hachage cryptographique prend une liste d'octets de taille arbitraire et renvoie une liste d'octets de taille fixe.

Cette fonction doit être

1. déterministe : une entrée donne toujours la même sortie
2. chaotique : des données similaires donnent des hachés distants
3. uniforme : pour des données aléatoires, tous les hachés sont équiprobables

In [ ]:
from hashlib import md5, sha256

### 6.1.a. Fonction `hash` de Python

À l'initialisation d'un script ou notebook, Python décide d'un sel aléatoire pour la fonction `hash`.

**TODO**  
Évaluer si la fonction `hash` de Python est déterministe. De même pour les hachages [MD5 et SHA-256 de `hashlib`](https://docs.python.org/3/library/hashlib.html#usage).

Pour ce faire, voici les résultats d'un hachage de `b"toto"` sur ma machine 

| algorithme | sortie |
| ---------- | ------ |
| `hash`     | 7220995997918355361 (int) |
| MD5        | f71dbe52628a3f83a77ab494817525c6 (hex) |
| SHA256     | MfemXjFVhqwZi9eYtmKc5JA9CJlHbVdBqfMuLlIbamY= (b64) |

In [ ]:
from base64 import b64encode

print("hash  :", hash(b"toto"))                    # change à chaque redémarrage du noyau
print("MD5   :", md5(b"toto").hexdigest())        # f71dbe52628a3f83a77ab494817525c6
print("SHA256:", b64encode(sha256(b"toto").digest()))   # MfemXjFV… comme dans le tableau

> La fonction `hash` de Python **n'est pas déterministe d'une session à l'autre** : elle est salée
> au démarrage (relancer le noyau change la valeur). Elle sert aux dictionnaires, pas à la cryptographie.
> **MD5 et SHA-256 sont déterministes** : on retrouve exactement les valeurs du tableau.

### 6.1.b. Collision MD5

Il y a trois degrés de sécurité pour une fonction de hachage :

1. PR : impossibilité de trouver une pré-image à un hash aléatoire
2. 2PR : impossibilité de trouver une pré-image qui donne le même hash que des données aléatoires
3. CR : impossibilité de trouver deux pré-images avec le même hash

La résistance aux collisions (CR) est le critère le plus restrictif et le plus couramment utilisé.

Trouver et exploiter des collisions est un domaine de recherche actif, notamment pour la falsification de certificats de sécurité. En théorie, Windows n'utilise plus de MD5 depuis 2013, notamment suite au [malware Flame](https://fr.wikipedia.org/wiki/Flame_(ver_informatique)), mais des failles utilisant MD5 ont néanmoins été [trouvées récemment](https://cendyne.dev/posts/2023-09-01-spoofing-certificates-with-md5-collisions.html).

**TODO**  
Comparer les checksums de `message1.bin` et `message2.bin` avec MD5 et SHA-256 dans une cellule, puis dans le terminal (`md5sum` et `sha256sum` pour les systèmes Unix, ["Get-FileHash"](https://learn.microsoft.com/en-us/powershell/module/microsoft.powershell.utility/get-filehash) sous Windows, ou bien [avec OpenSSL](https://docs.openssl.org/master/man1/openssl-dgst/#examples)).

Faire de même avec les images `plane.jpg` et `ship.jpg`.

In [ ]:
def empreintes(chemin):
    with open(chemin, "rb") as f:
        contenu = f.read()
    return md5(contenu).hexdigest(), sha256(contenu).hexdigest()

for a, b in [("message1.bin", "message2.bin"), ("plane.jpg", "ship.jpg")]:
    (ma, sa), (mb, sb) = empreintes(a), empreintes(b)
    print(f"{a} / {b} : MD5 identiques ? {ma == mb} · SHA-256 identiques ? {sa == sb}")

# Dans le terminal (Linux, macOS) :
!md5sum message1.bin message2.bin plane.jpg ship.jpg
!sha256sum message1.bin message2.bin plane.jpg ship.jpg
# Avec OpenSSL (tous systèmes) :
!openssl dgst -md5 message1.bin message2.bin

> Les deux paires de fichiers ont **la même empreinte MD5 mais des SHA-256 différents** :
> ce sont des **collisions MD5**. MD5 n'est plus résistant aux collisions (CR) ; SHA-256 l'est toujours.

## 6.2. HMAC

*Rappel :* L'objectif de la cryptographie est d'assurer la confidentialité et l'intégrité. Le chiffrement assure seulement la confidentialité.

Pour assurer l'intégrité d'un message, on peut y attacher un **MAC** (*Message Authentication Code*). Cela permet de valider la source et la non-modification du message.

Si ce MAC est basé sur une fonction de hachage, on parle de **HMAC**.

### 6.2.a. Padding de clé

La clé $K$ est pré-traitée pour calculer un HMAC.

**TODO**  
Compléter la fonction ci-dessous pour qu'elle corresponde à [`computeBlockSizedKey` de Wikipédia](https://en.wikipedia.org/wiki/HMAC#Implementation).

Faire le test sur la clé `"key"`.

In [ ]:
def key_pad(key: bytes, algo) -> bytes:
    bs = algo().block_size          # 64 octets pour MD5, SHA-1 et SHA-256
    n = len(key)
    if n > bs:                      # clé trop longue : on la remplace par son haché
        key = algo(key).digest()
    return key + b"\x00" * (bs - len(key))   # puis on complète avec des zéros


k = key_pad(b"key", md5)
print(len(k), k[:5])                         # 64 b'key\x00\x00'
print(len(key_pad(b"x" * 100, md5)))         # 64 aussi : clé longue hachée puis complétée

### 6.2.b. Calcul de HMAC

Une fois la clé pré-traitée, on définit 

$ \displaystyle\quad\qquad \begin{cases} K_\mathtt{o} = K \oplus \mathtt{opad} \\ K_\mathtt{i} = K \oplus \mathtt{ipad} \end{cases}$

avec $\mathtt{ipad} = \mathtt{0x36}$ et $\mathtt{opad} = \mathtt{0x5c}$, répétés autant que nécessaire. L'étiquette d'un message $t$ est alors calculée avec la formule

$ \displaystyle\quad\qquad \mathrm{HMAC}(K, t) = H\bigl( K_\mathtt{o} ~\|~ H (K_\mathtt{i} \| t) \bigr) $

**Attention :** En pratique, le MAC est calculé sur les données [déjà chiffrées](https://fr.wikipedia.org/wiki/Chiffrement_authentifi%C3%A9#Chiffrement-puis-MAC_(Encrypt-then-MAC,_ou_EtM)), et il n'utilise pas la même clé que le chiffrement.

**TODO**  
Implémenter ce calcul de HMAC puis effectuer des tests avec [les exemples de Wikipédia](https://en.wikipedia.org/wiki/HMAC#Examples).

In [ ]:
def hmac(key: bytes, tag: bytes, algo=md5):
    key = key_pad(key, algo)
    k_o = bytes(k ^ 0x5C for k in key)       # K ⊕ opad, opad répété sur tout le bloc
    k_i = bytes(k ^ 0x36 for k in key)       # K ⊕ ipad
    interne = algo(k_i + tag).digest()       # H(K_i ‖ t)
    return algo(k_o + interne).digest()      # H(K_o ‖ H(K_i ‖ t))


# Exemples de Wikipédia (clé "key")
t = b"The quick brown fox jumps over the lazy dog"
print(hmac(b"key", t, md5).hex()    == "80070713463e7749b90c2dc24911e275")
print(hmac(b"key", t, sha256).hex() == "f7bc83f430538424b13298e6aa6fb143ef4d59a14946175997479dbc2d1a3cd8")

# Contre-vérification avec le module standard
import hmac as hmac_std
print(hmac(b"key", t, sha256) == hmac_std.new(b"key", t, sha256).digest())

### 6.2.c. Comparaison avec OpenSSL

On peut aussi utiliser OpenSSL pour [calculer des MAC](https://docs.openssl.org/master/man1/openssl-mac/#examples).

**TODO**  
Exécuter la commande suivante et valider le résultat avec votre fonction.

```sh
openssl mac -digest MD5 \
    -macopt hexkey:000102030405060708090A0B0C0D0E0F \
    -in message1.bin HMAC
```

In [ ]:
!openssl mac -digest MD5 -macopt hexkey:000102030405060708090A0B0C0D0E0F -in message1.bin HMAC

with open("message1.bin", "rb") as f:
    m1 = f.read()
print(hmac(bytes.fromhex("000102030405060708090A0B0C0D0E0F"), m1, md5).hex().upper())
# Les deux lignes doivent afficher le même code (OpenSSL l'écrit en majuscules).

## 6.3. Hachage de mots de passe

La fonction de hachage Argon2 est spécialisée dans le chiffrement de mots de passe. 

Nous allons l'étudier brièvement et comparer son utilisation avec SHA-256.

### 6.3.a. Hachage Argon2

Cette fonction de hachage est incluse dans [un module Python](https://argon2-cffi.readthedocs.io/en/stable/).

Avec la fonction `hash`, la sortie est une concaténation (séparée par des `$`) de 
- l'algorithme
- la version
- les paramètres (coût mémoire `m`, coût en temps `t` et parallélisation `p`)
- le sel en Base64
- le haché en Base64

On peut les récupérer en utilisant la méthode `str.split("$")`.

**TODO**  
Chiffrer un mot de passe. En octets, quelle est la taille du sel ? Quelle est la taille du haché ?

In [ ]:
import argon2
from base64 import b64decode

ph = argon2.PasswordHasher()
h = ph.hash("P@5sword!")
print(h)
_, algo, version, params, sel_b64, hache_b64 = h.split("$")   # la chaîne commence par "$"

def b64_sans_padding(s):
    return b64decode(s + "=" * (-len(s) % 4))   # Argon2 omet les "=" finaux

print("sel :", len(b64_sans_padding(sel_b64)), "octets")      # 16
print("haché :", len(b64_sans_padding(hache_b64)), "octets")  # 32

### 6.3.b. Vérification Argon2

**TODO**  
Vérifier que le résultat 

```txt
$argon2id$v=19$m=65536,t=3,p=4$All+Them+Witches$aIFMB1pUf+7vQ/Q0WdUdT9l5iotOCrOu0nGQLR2ANP0
```

correspond bien au mot de pase `P@5sword!`.

In [ ]:
pwd_test = "P@5sword!"
hash_test = "$argon2id$v=19$m=65536,t=3,p=4$QUxMIFRIRU0gV0lUQ0hFUw$7rDvATKFD+m+e3OpLwWdbICLuv7zbJYLqqetwmswVbk"

print(ph.verify(hash_test, pwd_test))     # True (lève VerifyMismatchError sinon)
print(b64_sans_padding("QUxMIFRIRU0gV0lUQ0hFUw"))   # le sel est b'ALL THEM WITCHES'

# La chaîne de l'énoncé (sel "All+Them+Witches") est valide elle aussi :
print(ph.verify("$argon2id$v=19$m=65536,t=3,p=4$All+Them+Witches$aIFMB1pUf+7vQ/Q0WdUdT9l5iotOCrOu0nGQLR2ANP0", pwd_test))

> Le mot de passe correspond. Le sel `QUxMIFRIRU0gV0lUQ0hFUw` est `ALL THEM WITCHES` en Base64.
> L'énoncé et la cellule ne donnent pas la même chaîne (sels différents), mais les deux sont valides.

### 6.3.c. Mots de passe fréquents

On a haché 3 mots de passe aléatoires dans [une liste des 207 mdp les plus communs](https://github.com/danielmiessler/SecLists/blob/master/Passwords/Common-Credentials/probable-v2_top-207.txt).

**TODO**  
Retrouver les mots de passe correspondants.

In [ ]:
with open("hashes-top-207.txt", "r") as f:
    hashes = f.read().splitlines()

with open("probable-v2_top-207.txt", "r") as f:
    passwords = f.read().splitlines()

# Attaque par dictionnaire : on essaie chaque mot de passe courant sur chaque haché.
# Le sel est lu dans le haché lui-même par verify().
for h in hashes:
    for pwd in passwords:
        try:
            ph.verify(h, pwd)
            print("trouvé :", pwd)
            break
        except argon2.exceptions.VerifyMismatchError:
            pass

> Les trois mots de passe sont **`batman`**, **`arsenal`** et **`chester`**, dans l'ordre des hachés.
> Le sel empêche les tables pré-calculées, mais pas l'attaque par dictionnaire : un mot de passe courant reste faible.

### 6.3.d. Comparaison avec SHA256

On fixe le mot de passe `"P@5sword!"` et le sel `"ALL THEM WITCHES"`.

**TODO**  
Avec la commande magique `%%timeit`, comparer le temps de calcul pour hacher un mdp avec Argon2 et avec SHA-256. Pour SHA-256, chiffrer la concaténation `sel + pwd`. Lequel serait le plus facile à *brute force* ?

In [ ]:
pwd = "P@5sword!"
salt = b"ALL THEM WITCHES"

%timeit sha256(salt + pwd.encode()).digest()
%timeit ph.hash(pwd, salt=salt)

> Ordres de grandeur : **≈ 0,4 µs pour SHA-256** contre **≈ 0,1 s pour Argon2**, soit plus de 100 000 fois plus lent.
> **SHA-256 est bien plus facile à attaquer en force brute.** Argon2 est lent et gourmand en mémoire (64 Mio)
> **exprès** : chaque essai coûte cher à l'attaquant, alors que l'utilisateur légitime ne paie qu'une fois.

## 6.A. Pour aller plus loin



### A.1. Clés de session

La fonction HMAC peut servir de générateur aléatoire pour dériver des clés de session.

Un algorithme courant est [**PBKDF2**](https://www.rfc-editor.org/rfc/rfc2898#section-5.2). Cela permet de convertir un secret partagé (avec Diffie-Hellman) en clé de chiffrement.

**TODO**  
Vérifier l'implémentation de PBKDF de `hashlib` avec [les cas test standards](https://www.rfc-editor.org/rfc/rfc6070#section-2).

Comparer avec `openssl kdf`:

`openssl kdf -keylen 20 -kdfopt digest:SHA1 -kdfopt pass:password -kdfopt salt:salt -kdfopt iter:2 PBKDF2`

In [ ]:
import hashlib

# RFC 6070 : P = "password", S = "salt", dkLen = 20, SHA-1
vecteurs = {1: "0c60c80f961f0e71f3a9b524af6012062fe037a6",
            2: "ea6c014dc72d6f8ccd1ed92ace1d41f0d8de8957",
            4096: "4b007901b765489abead49d926f721d065a429c1"}
for iterations, attendu in vecteurs.items():
    obtenu = hashlib.pbkdf2_hmac("sha1", b"password", b"salt", iterations, 20).hex()
    print(iterations, obtenu == attendu)

!openssl kdf -keylen 20 -kdfopt digest:SHA1 -kdfopt pass:password -kdfopt salt:salt -kdfopt iter:2 PBKDF2

### A.2. AES GCM

Beaucoup de connexions sont sécurisés avec la suite cryptographique « TLS\_AES\_128\_GCM\_SHA256 ». C'est le cas par exemple de Wikipédia.

Le mode GCM (Galois Counter Mode) est un mode de chiffrement **authentifié**. En plus de l'IV (appelé *nonce*) de 96 bits, il renvoie un MAC.

Attention, le *nonce* ne doit [**jamais**](https://frereit.de/aes_gcm/) être réutilisé avec une clé donnée. Sinon il devient très facile de falsifier le MAC.

**TODO**  
Utiliser le module `cryptography` pour chiffrer un message avec AES-GCM. Est-ce qu'il y a besoin de padding ? Comment récupérer le MAC ?

Déchiffrer les données (en b64) `sMfhRF5NdNuS4ZipugVPho0UdW0sjF1yAuRg` et `sMfhRF5NdNuS4ZipugVPho0UdW0sjF1yAuRh` qui ont été chiffrées avec la clé `"ALL THEM WITCHES"` et le nonce `"KNOWER 4EVER"` (pas d'AAD).

*Remarque :* Ce chiffrement n'est malheureusement [pas disponible](https://crypto.stackexchange.com/questions/76177/how-to-encrypt-files-using-aes256-gcm-cipher-under-linux) avec OpenSSL en ligne de commande.

In [ ]:
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.exceptions import InvalidTag
from base64 import b64decode

aes = AESGCM(b"ALL THEM WITCHES")
nonce = b"KNOWER 4EVER"                         # 12 octets = 96 bits

c = aes.encrypt(nonce, b"bonjour", None)        # None : pas de données associées (AAD)
print(len(c), "octets : 7 de message + 16 de MAC")
print("MAC :", c[-16:].hex())                   # le MAC est collé à la fin

for donnees in ["sMfhRF5NdNuS4ZipugVPho0UdW0sjF1yAuRg", "sMfhRF5NdNuS4ZipugVPho0UdW0sjF1yAuRh"]:
    try:
        print(aes.decrypt(nonce, b64decode(donnees), None))
    except InvalidTag:
        print("InvalidTag : message modifié, déchiffrement refusé")

> **Pas besoin de padding** : GCM est un mode « compteur », il transforme AES en chiffrement par flot.
> Le MAC (16 octets) est **concaténé à la fin** du chiffré renvoyé par `encrypt`.
> Le premier message donne `b'Flower Face'`. Le second ne diffère que du dernier caractère : le MAC
> ne correspond plus et `decrypt` lève `InvalidTag`. C'est l'**intégrité** garantie par le chiffrement authentifié.

### A.3. Construction de Merkle-Damgård

Le hachage MD5 est basé sur une fonction de compression $f$ qui prend deux blocs de 64 octets et n'en renvoie qu'un.

Le haché final est alors calculé en découpant les données $x$ en blocs de 64 octets $x = x_1 \| x_2 \| ... \| x_N$ (avec *padding*) et en posant

$\displaystyle\quad\qquad\begin{cases} H_0 = IV \text{~~de 64 octets}, \\ H_i = f(H_{i-1}, x_i) \end{cases}$

Le haché est alors $\mathrm{MD5}(x_1 \| x_2 \| ... \| x_N) = H_N$. Le diagramme suivant (source [Wikipédia](https://commons.wikimedia.org/wiki/File:Merkle-Damgard-padding-fr.svg?uselang=fr)) illustre cela :

![](Merkle-Damgard-padding-fr.svg)

Le hachage SHA2 (le plus répandu aujourd'hui) utilise la même construction, avec des tailles différentes.

**TODO**  
Comparer le haché de la liste vide avec l'IV de MD5

```txt
67 45 23 01 ef cd ab 89 98 ba dc fe 10 32 54 76
```

Comment expliquer cette différence ?

Que se passe-t-il si on rajoute un préfixe à notre collision ? Et un suffixe ?

In [ ]:
print("MD5('') :", md5(b"").hexdigest())
print("IV      :", "67452301efcdab8998badcfe10325476")

with open("message1.bin", "rb") as f:
    m1 = f.read()
with open("message2.bin", "rb") as f:
    m2 = f.read()

suffixe = b"un suffixe identique"
prefixe = b"P" * 64                                     # un bloc entier ajouté devant
print("suffixe : collision ?", md5(m1 + suffixe).digest() == md5(m2 + suffixe).digest())   # True
print("préfixe : collision ?", md5(prefixe + m1).digest() == md5(prefixe + m2).digest())   # False en général

> `MD5('') = d41d8cd98f00b204e9800998ecf8427e` n'est pas l'IV : même vide, le message est **paddé**
> (un bit 1, des zéros, puis sa longueur sur 64 bits) pour former un bloc. La fonction de compression
> est donc appliquée une fois, H₁ = f(IV, bloc de padding).
>
> **Suffixe** : la collision **reste** une collision. Après les blocs en collision, l'état interne est le même
> des deux côtés (et les longueurs sont égales). Les mêmes blocs ajoutés donnent donc le même haché.
>
> **Préfixe** : la collision est en général **perdue**. Elle a été calculée pour un état de départ précis
> (l'IV) ; un préfixe change cet état d'entrée. Il faudrait recalculer une collision à partir de ce nouvel état.